# 01 · Ingestão — camada Bronze

*Do sitemap (via GitHub Actions) para a tabela bronze_articles*

> **FIAP Pós-Tech · Fase 5 (MLOps)** — Recomendação de notícias com MLOps no Databricks.
> Matérias **reais** (sitemap do O Globo) · usuários e cliques **sintéticos**.

## De onde vêm as matérias?

Do **sitemap de notícias** do O Globo (`news.xml`): URL, título, data e imagem das
~700 matérias das últimas 48 h. A **editoria** sai da própria URL
(`/economia/...`, `/esportes/futebol/...`).

**Quem baixa é o GitHub Actions**, não o Databricks: na Free Edition a internet de saída
é restrita. O workflow `ingest.yml` roda todo dia, grava um JSON e envia para o
**Volume** `raw/catalog/`. Este notebook lê o que estiver lá (e, se o Volume estiver
vazio, usa o snapshot `data/articles_snapshot.json` que vem no bundle).

Guardamos **só metadados** — o app linka para o site original.

In [ ]:
# Célula 1 — dependências (sempre a primeira: o restartPython apaga as variáveis)
%pip install -q -r ../requirements-databricks.txt
dbutils.library.restartPython()

In [ ]:
# Célula 2 — src/ no sys.path + widgets (= parâmetros do Job) -> settings
import sys
from pathlib import Path


def _find_src() -> Path:
    candidates = [Path.cwd().parent / "src"]
    try:  # fallback: caminho do notebook no workspace
        ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
        candidates.append(Path("/Workspace" + ctx.notebookPath().get()).parent.parent / "src")
    except Exception:  # noqa: BLE001
        pass
    for c in candidates:
        if (c / "news_recsys").is_dir():
            return c.resolve()
    raise FileNotFoundError(f"src/ não encontrado em {candidates}")


SRC = _find_src()
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
ROOT = SRC.parent

from news_recsys.config import Settings

dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("schema", "news_recsys_dev")
dbutils.widgets.text("n_users", "300")
dbutils.widgets.text("seed", "42")

settings = Settings(_env_file=None).with_overrides(
    catalog=dbutils.widgets.get("catalog"),
    schema_name=dbutils.widgets.get("schema"),
    n_users=dbutils.widgets.get("n_users"),
    seed=dbutils.widgets.get("seed"),
)
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {settings.catalog}.{settings.schema_name}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {settings.catalog}.{settings.schema_name}.{settings.volume}")
print(f"schema ..... {settings.catalog}.{settings.schema_name}")
print(f"modelo ..... {settings.model_fqn}")
print(f"usuários ... {settings.n_users} (seed={settings.seed})")

### `features/sections.py` e `ingestion/catalog.py`

In [ ]:
# src/news_recsys/features/sections.py  (somente leitura — o código de verdade está em src/)
"""Editoria (section) e subeditoria a partir da URL da matéria.

Ex.: ``https://oglobo.globo.com/esportes/futebol/noticia/...`` → section ``esportes``,
subsection ``esportes/futebol``.
"""

from __future__ import annotations

from urllib.parse import urlparse

KNOWN_SECTIONS: frozenset[str] = frozenset(
    {
        "blogs",
        "brasil",
        "ciencia",
        "cultura",
        "economia",
        "ela",
        "esportes",
        "mundo",
        "opiniao",
        "patrocinado",
        "politica",
        "rio",
        "saude",
        "tecnologia",
    }
)
# Seções "satélite" mapeadas para a editoria principal
SECTION_ALIASES: dict[str, str] = {
    "apuracao-eleicoes-2026": "politica",
    "play": "cultura",
    "rioshow": "cultura",
}
# Conteúdo patrocinado não entra no catálogo recomendável
NON_RECOMMENDABLE: frozenset[str] = frozenset({"patrocinado"})
OTHER = "outros"
_STOP_SEGMENTS = {"noticia", "post", "coluna", "video", "podcast"}


def _segments(url: str) -> list[str]:
    return [s for s in urlparse(url).path.split("/") if s]


def section_from_url(url: str) -> str:
    segments = _segments(url)
    if not segments:
        return OTHER
    first = segments[0].lower()
    first = SECTION_ALIASES.get(first, first)
    return first if first in KNOWN_SECTIONS else OTHER


def subsection_from_url(url: str) -> str:
    """``<section>/<segundo segmento>`` quando é subeditoria; senão só a section."""
    section = section_from_url(url)
    segments = _segments(url)
    if len(segments) < 2:
        return section
    second = segments[1].lower()
    if second in _STOP_SEGMENTS or second.isdigit() or second.endswith(".ghtml"):
        return section
    return f"{section}/{second}"


def is_recommendable(section: str) -> bool:
    return section not in NON_RECOMMENDABLE

In [ ]:
# src/news_recsys/ingestion/catalog.py  (somente leitura — o código de verdade está em src/)
"""Catálogo de matérias: sitemap → registros com id estável e editoria."""

from __future__ import annotations

import hashlib
import json
from collections.abc import Iterable, Sequence
from datetime import UTC, datetime
from pathlib import Path
from typing import Any

import requests

from news_recsys.features.sections import (
    is_recommendable,
    section_from_url,
    subsection_from_url,
)
from news_recsys.ingestion.sitemap import ArticleMeta, parse_sitemap

USER_AGENT = "Mozilla/5.0 (compatible; news-recsys-aula/0.1)"
ARTICLE_FIELDS = (
    "article_id",
    "url",
    "title",
    "section",
    "subsection",
    "published_at",
    "image_url",
    "first_seen_at",
)


def article_id_from_url(url: str) -> str:
    """Id curto e estável (a URL é a chave natural; o hash só encurta)."""
    clean = url.split("?", 1)[0].split("#", 1)[0].rstrip("/")
    return "a_" + hashlib.sha1(clean.encode("utf-8")).hexdigest()[:12]  # noqa: S324


def to_record(meta: ArticleMeta, first_seen_at: str) -> dict[str, Any]:
    return {
        "article_id": article_id_from_url(meta.url),
        "url": meta.url,
        "title": meta.title,
        "section": section_from_url(meta.url),
        "subsection": subsection_from_url(meta.url),
        "published_at": meta.published_at,
        "image_url": meta.image_url,
        "first_seen_at": first_seen_at,
    }


def build_records(metas: Iterable[ArticleMeta], now: str | None = None) -> list[dict[str, Any]]:
    """Registros do catálogo — sem título ou patrocinados ficam de fora."""
    ts = now or datetime.now(UTC).isoformat(timespec="seconds")
    records = [to_record(m, ts) for m in metas if m.title]
    return [r for r in records if is_recommendable(r["section"])]


def fetch_sitemap(url: str, timeout: float = 20.0) -> str:
    response = requests.get(url, headers={"User-Agent": USER_AGENT}, timeout=timeout)
    response.raise_for_status()
    return response.text


def merge_records(
    existing: Sequence[dict[str, Any]], new: Sequence[dict[str, Any]]
) -> list[dict[str, Any]]:
    """Une catálogos por ``article_id``; mantém o ``first_seen_at`` mais antigo."""
    merged: dict[str, dict[str, Any]] = {r["article_id"]: dict(r) for r in existing}
    for record in new:
        old = merged.get(record["article_id"])
        if old is None:
            merged[record["article_id"]] = dict(record)
        else:
            merged[record["article_id"]] = {**record, "first_seen_at": old["first_seen_at"]}
    return sorted(merged.values(), key=lambda r: (r["published_at"], r["article_id"]))


def load_catalog(path: Path) -> list[dict[str, Any]]:
    if not Path(path).exists():
        return []
    payload = json.loads(Path(path).read_text(encoding="utf-8"))
    return list(payload["articles"] if isinstance(payload, dict) else payload)


def save_catalog(records: Sequence[dict[str, Any]], path: Path) -> int:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    payload = {
        "source": "oglobo-news-sitemap",
        "generated_at": datetime.now(UTC).isoformat(timespec="seconds"),
        "n_articles": len(records),
        "articles": list(records),
    }
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=1), encoding="utf-8")
    return len(records)


def fetch_catalog(url: str, now: str | None = None) -> list[dict[str, Any]]:
    """Sitemap ao vivo → registros do catálogo."""
    return build_records(parse_sitemap(fetch_sitemap(url)), now=now)

### Execução: lendo os JSONs do Volume

In [ ]:
import json

import pandas as pd

from news_recsys.ingestion.catalog import load_catalog, merge_records

raw_dir = Path(settings.volume_path("catalog"))
raw_dir.mkdir(parents=True, exist_ok=True)
files = sorted(raw_dir.glob("*.json"))
if not files:  # Volume vazio (primeira execução): usa o snapshot do repositório
    files = [ROOT / "data" / "articles_snapshot.json"]
    print("Volume vazio — usando o snapshot do bundle:", files[0].name)

records: list[dict] = []
for f in files:
    records = merge_records(records, load_catalog(f))
    print(f"{f.name:40s} acumulado: {len(records)} matérias")

catalog_pdf = pd.DataFrame(records)
catalog_pdf["section"].value_counts().rename("matérias").to_frame()

### Execução: `MERGE` na `bronze_articles`

O catálogo **acumula** dia após dia. `MERGE` é o *upsert* do Delta: atualiza quem já
existe (pela chave `article_id`) e insere o que é novo — sem duplicar.

In [ ]:
from pyspark.sql.types import StringType, StructField, StructType

BRONZE = settings.table("bronze_articles")
cols = ["article_id", "url", "title", "section", "subsection", "published_at", "image_url", "first_seen_at"]
schema = StructType([StructField(c, StringType(), True) for c in cols])
spark.createDataFrame(catalog_pdf[cols], schema=schema).createOrReplaceTempView("novas_materias")

spark.sql(f"CREATE TABLE IF NOT EXISTS {BRONZE} ({', '.join(f'{c} STRING' for c in cols)}, ingested_at TIMESTAMP)")
insert_cols = ", ".join(cols)
insert_vals = ", ".join(f"s.{c}" for c in cols)
result = spark.sql(
    f"""
    MERGE INTO {BRONZE} AS t
    USING novas_materias AS s
    ON t.article_id = s.article_id
    WHEN MATCHED THEN UPDATE SET t.title = s.title, t.image_url = s.image_url, t.ingested_at = current_timestamp()
    WHEN NOT MATCHED THEN INSERT ({insert_cols}, ingested_at) VALUES ({insert_vals}, current_timestamp())
    """
)
result.toPandas()

In [ ]:
print(BRONZE, "->", spark.table(BRONZE).count(), "matérias")
spark.table(BRONZE).select("section", "title", "published_at").orderBy("published_at", ascending=False).limit(8).toPandas()

### O Delta guarda o histórico de cada escrita

In [ ]:
spark.sql(f"DESCRIBE HISTORY {BRONZE}").select("version", "timestamp", "operation", "operationMetrics").limit(5).toPandas()

### O que percebemos?

- Coleta (GitHub Actions) e processamento (Databricks) estão **desacoplados** pelo Volume.
- `MERGE` torna a ingestão **idempotente**: rodar duas vezes não duplica nada.
- `DESCRIBE HISTORY` = auditoria grátis (dá até para consultar a tabela "como era ontem").

**Próximo:** `02_eventos_sinteticos`.